In [3]:
import os
import sys
import pprint
from pathlib import Path
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.teams import SelectorGroupChat, RoundRobinGroupChat
from autogen_agentchat.tools import AgentTool
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools

# Project import paths
# Get the current directory (which is TFM_BA_GPT/test)
notebook_dir = os.getcwd()

# Define base project directory (one level up from test/)
project_root = os.path.abspath(os.path.join(notebook_dir, '..'))

# Server root: one level up from project_root
code_root = os.path.abspath(os.path.join(project_root, '..'))
server_root = os.path.join(project_root, 'mcp_servers')



# Define paths to each server and Python executable
PyMCP_FS_path = os.path.join(server_root, 'PyMCP-FS')
Mermaid_MCP_path = os.path.join(server_root, 'Mermaid-MCP')
PDF_format_path = os.path.join(server_root, 'Final_deliverable_MCP_server')
tmp_dir = os.path.join(server_root, 'tmp')

# Python executables
PyMCP_FS_python = os.path.join(PyMCP_FS_path, '.venv', 'bin', 'python')
Mermaid_MCP_python = os.path.join(Mermaid_MCP_path, 'venv', 'bin', 'python')
PDF_format_python = os.path.join(PDF_format_path, 'venv', 'bin', 'python')




# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

In [4]:
model_client = load_model_client("openai")

In [13]:
# Helpful sanity prints:
print("raw:", getattr(model_client, "_raw_config", None))
print("create_args:", getattr(model_client, "_create_args", None))

raw: {'model': 'gpt-4o', 'api_key': 'REPLACE_WITH_YOUR_API_KEY'}
create_args: {'model': 'gpt-4o'}


In [6]:
# Server params
PyMCP_FS = StdioServerParams(
    command="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.venv/bin/python",
    args=[
        "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/mcp_servers/PyMCP-FS/main.py",
        "-d", project_root,      # ✅
        "-d", "/tmp",            # ✅ was missing the -d
        "--log-level", "DEBUG",
        "--log-file", "mcp_debug.log",
        "--project-root", project_root,
    ],
)

In [7]:


Mermaid_MCP = StdioServerParams(
    command="uv",
    args=[
        "run",
        os.path.join(Mermaid_MCP_path, 'Mermaid-MCP.py'),
        '-d', project_root,
        '-d', tmp_dir,
        '--project-root', project_root
    ]
)

PDF_format_mcp_server = StdioServerParams(
    command="uv",
    args=[
        "run",
        os.path.join(PDF_format_path, 'PDF_format_mcp_server.py'),
        '-d', project_root,
        '--project-root', project_root,
    ],
    read_timeout_seconds=60.0,
)


In [8]:
PyMCP_FS_mcp_tools = await mcp_server_tools(PyMCP_FS)

In [9]:
mermaid_mcp_tools = await mcp_server_tools(Mermaid_MCP)

In [10]:
pdf_format_mcp_tools = await mcp_server_tools(PDF_format_mcp_server)

In [11]:

# ---- terminate_agent system message (MOVE + clean + render_mermaid_all + robust PDF) ----
terminate_system_message = r"""
You are the TERMINATE_AGENT. Perform the final export pipeline and then end with **TERMINATE**.

**INPUTS**
  conversation log: project_root/conversation/interview.jsonl
  spec:             project_root/conversation/functional_specification.md
  mermaid:          project_root/conversation/process_diagram.mmd

**GOAL**
  Produce, inside project_root/conversation/<SESSION_ID>/:
      interview.jsonl
      functional_specification.md
      process_diagram.mmd
      process_diagram.png
      functional_specification.formatted.md
      deliverable.pdf

**STEPS STRICT ORDER**
  1 Call 'read_md_file_tool' with file_path="conversation/interview.jsonl" and extract the first available "session_id" from its content.
    If multiple lines: pick the first non-empty "session_id".
    If not found or the read fails: set SESSION_ID = "sess-unknown".

  2 Create folder project_root/conversation/<SESSION_ID>/ (ensure_dir).

  3 Clean the Mermaid file **in place** and report issues (severity/code/message/recommendation):
      clean_mermaid(
        input_path="project_root/conversation/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/conversation/<SESSION_ID>/process_diagram.mmd",
        overwrite=True
      )

  4 Render Mermaid **PNG**:
      render_mermaid_png(
        input_path="project_root/conversation/<SESSION_ID>/process_diagram.mmd",
        output_path="project_root/conversation/<SESSION_ID>/process_diagram.png",
        theme="forest"
      )

  5 Run **format_and_export** ONLY:
      format_and_export(
        session_dir="project_root/conversation/<SESSION_ID>",
        md="functional_specification.md",
        diagram="process_diagram.png",
        md_out="functional_specification.formatted.md",
        pdf_out="deliverable.pdf",
        toc=True, toc_depth=2, margin="1in"
      )

  6 Confirm project_root/conversation/<SESSION_ID>/deliverable.pdf exists.

  7 Save experience metadata:
      save_experience_chroma(
        session_dir="project_root/conversation/<SESSION_ID>",
        session_id="<SESSION_ID>"
      )

**RULES**
  Be explicit about each steps success or failure.
  If a tool call fails, retry up to 1 times with the same arguments.
"""


In [12]:
termination = TextMentionTermination("TERMINATE")

terminate_agent = AssistantAgent(
    name="terminate_agent",
    model_client=model_client,
    system_message=terminate_system_message,
    tools=(PyMCP_FS_mcp_tools + mermaid_mcp_tools + pdf_format_mcp_tools),
    reflect_on_tool_use=False,
)
team = RoundRobinGroupChat(
    [terminate_agent],
    termination_condition=termination,
    )
await Console(team.run_stream(task="Execute the termination pipeline"))

---------- TextMessage (user) ----------
Execute the termination pipeline
---------- ToolCallRequestEvent (terminate_agent) ----------
[FunctionCall(id='call_aUpSySTQ2XRT3NFYMSUkgTJz', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')]
---------- ToolCallExecutionEvent (terminate_agent) ----------
[FunctionExecutionResult(content='[{"type": "text", "text": "{\\"_meta\\": {\\"session_id\\": \\"sess-20231119122300\\", \\"ts\\": \\"2025-09-19T06:26:26.408091Z\\", \\"n_records\\": 2}}\\n{\\"speaker\\": \\"interviewer\\", \\"message\\": \\"QUESTION: Can you provide an overview of the new product you have in mind?\\", \\"subject\\": \\"Product Overview\\", \\"q_index\\": 1, \\"timestamp\\": \\"2025-09-19T06:26:26.408058Z\\"}\\n{\\"speaker\\": \\"user\\", \\"message\\": \\"The scope of the project is to design and develop an online ecommerce website for apparels. Customers will be able to use the frontend website for searching the apparels and place the order

TaskResult(messages=[TextMessage(id='65e1fcef-8ead-4cf7-b378-84574d769d11', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 6, 44, 5, 901624, tzinfo=datetime.timezone.utc), content='Execute the termination pipeline', type='TextMessage'), ToolCallRequestEvent(id='7f6d337c-7d8a-452e-9528-08d4e46be716', source='terminate_agent', models_usage=RequestUsage(prompt_tokens=3288, completion_tokens=21), metadata={}, created_at=datetime.datetime(2025, 9, 19, 6, 44, 8, 292963, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_aUpSySTQ2XRT3NFYMSUkgTJz', arguments='{"path":"project_root/conversation/interview.jsonl"}', name='read_file')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='bba8b955-6ff0-4304-9881-b5313161abf3', source='terminate_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2025, 9, 19, 6, 44, 9, 119915, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='[{"type": "text", "text": "